# Tarea 7 - Sistemas dinámicos

## 1. Sistema masa-resorte-amortiguador

### b) Trayectoria en el plano posición–velocidad

Se considera el sistema con $$m=k=1$$ y $$c=3$$, con condición inicial

$$
\begin{pmatrix}u(0)\\v(0)\end{pmatrix}
=
\begin{pmatrix}1\\1\end{pmatrix},
\qquad v=\dot{u}.
$$

El sistema de primer orden es

$$
\frac{d}{dt}\begin{pmatrix}u\\v\end{pmatrix}
=
\begin{pmatrix}0&1\\-1&-3\end{pmatrix}
\begin{pmatrix}u\\v\end{pmatrix}.
$$

Se determinarán los coeficientes de la solución modal y se graficará la trayectoria $$v$$ vs. $$u$$ desde $$t=0$$ hasta aproximar $$t\to\infty$$.

### Autovalores, autovectores y coeficientes

El polinomio característico es

$$
\lambda^2+3\lambda+1=0,
\qquad
\lambda_{1,2}=\frac{-3\pm\sqrt{5}}{2}.
$$

Para cada autovalor, un autovector es $$\mathbf{q}_i=(1,\lambda_i)^T$$. Por tanto, la solución tiene la forma

$$
\begin{pmatrix}u(t)\\v(t)\end{pmatrix}
=
\alpha_1 e^{\lambda_1t}\begin{pmatrix}1\\\lambda_1\end{pmatrix}
+
\alpha_2 e^{\lambda_2t}\begin{pmatrix}1\\\lambda_2\end{pmatrix}.
$$

Los coeficientes se calculan imponiendo la condición inicial $$\mathbf{x}(0)=(1,1)^T$$. El código también verifica numéricamente que la reconstrucción en $$t=0$$ coincide con el dato inicial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.ticker import MultipleLocator

# Parámetros del sistema
m = 1.0
k = 1.0
c = 3.0
H = np.array([[0.0, 1.0], [-k / m, -c / m]])
x0 = np.array([1.0, 1.0])  # [u(0), v(0)]

# Autovalores y autovectores, ordenados de mayor a menor
lambdas, Q = np.linalg.eig(H)
order = np.argsort(lambdas)[::-1]
lambdas = lambdas[order]
Q = Q[:, order]

# Normalizamos cada autovector para que su primera componente sea 1
Q = Q / Q[0, :]

# x(0) = Q alpha, de modo que alpha = Q^{-1} x(0)
alpha = np.linalg.solve(Q, x0)

print("Autovalores:")
for i, lam in enumerate(lambdas, start=1):
    print(f"lambda_{i} = {lam:.6f}")
print("\nAutovectores (columnas de Q):")
print(Q)
print("\nCoeficientes modales:")
for i, a in enumerate(alpha, start=1):
    print(f"alpha_{i} = {a:.6f}")
print("\nEstado reconstruido en t = 0:", Q @ alpha)

### Gráfica de la trayectoria de fase

La solución modal se evalúa en un intervalo suficientemente largo para que la trayectoria se acerque al equilibrio $$ (u,v)=(0,0) $$. El color indica el tiempo: la curva comienza en $$ (1,1) $$ y, al avanzar el tiempo, converge al origen.

In [ ]:
# Evolución temporal de la solución modal
t = np.linspace(0.0, 20.0, 1400)
X = Q @ (alpha[:, None] * np.exp(lambdas[:, None] * t))
u = X[0, :]
v = X[1, :]

# Curva con color gradual para indicar el paso del tiempo
points = np.column_stack((u, v)).reshape(-1, 1, 2)
segments = np.concatenate((points[:-1], points[1:]), axis=1)
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
line = LineCollection(
    segments,
    cmap="viridis",
    norm=Normalize(t.min(), t.max()),
    linewidth=2.8,
)
line.set_array(t[:-1])
ax.add_collection(line)

# Marcar el estado inicial y el equilibrio al que converge
ax.scatter(u[0], v[0], s=90, color="#e76f51", edgecolor="white", linewidth=1.2,
           zorder=4, label=r"Inicio: $(u(0),v(0))=(1,1)$")
ax.scatter(0, 0, s=100, marker="*", color="#264653", edgecolor="white", linewidth=0.8,
           zorder=5, label=r"Equilibrio: $(0,0)$")

# Flechas sobre la curva para mostrar el sentido de evolución
for idx in (70, 250, 470, 700, 940, 1160):
    ax.annotate(
        "",
        xy=(u[idx + 18], v[idx + 18]),
        xytext=(u[idx], v[idx]),
        arrowprops=dict(arrowstyle="-|>", color="#343a40", lw=1.4, mutation_scale=13),
        zorder=3,
    )

ax.set_xlim(-0.08, 1.18)
ax.set_ylim(-0.72, 1.12)
ax.set_xlabel(r"Posición, $u(t)$", fontsize=12)
ax.set_ylabel(r"Velocidad, $v(t)=\dot{u}(t)$", fontsize=12)
ax.set_title(r"Trayectoria de fase para $m=k=1$, $c=3$", fontsize=15, pad=12)
ax.xaxis.set_major_locator(MultipleLocator(0.2))
ax.yaxis.set_major_locator(MultipleLocator(0.2))
ax.minorticks_on()
ax.grid(which="major", linewidth=0.75, alpha=0.65)
ax.grid(which="minor", linewidth=0.35, alpha=0.35)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
ax.axvline(0, color="black", linewidth=0.8, alpha=0.7)
ax.legend(loc="upper right", frameon=True)
colorbar = fig.colorbar(line, ax=ax, pad=0.02)
colorbar.set_label(r"Tiempo, $t$")
plt.show()

### Interpretación

La trayectoria inicia en $$ (u,v)=(1,1) $$. Como la velocidad inicialmente es positiva, la posición primero aumenta; después, la velocidad cambia de signo y el sistema regresa hacia el equilibrio. Ambos autovalores son negativos, de modo que

$$
\lim_{t\to\infty}u(t)=0,
\qquad
\lim_{t\to\infty}v(t)=0.
$$

El movimiento es sobreamortiguado y la trayectoria converge al origen sin oscilaciones repetidas.


### d) Construcción y gráfica de soluciones reales espirales

Para $$m=k=1$$, la matriz del sistema es

$$
H=\begin{pmatrix}0&1\\-1&-c\end{pmatrix}.
$$

Cuando $$0<c<2$$, los autovalores son conjugados:

$$
\lambda_{1,2}=a\pm ib,
\qquad a=-\frac{c}{2},
\qquad b=\frac{\sqrt{4-c^2}}{2}.
$$

Usando el autovector $$\mathbf{Y}_1=(1,a+ib)^T=\mathbf{U}_R+i\mathbf{U}_I$$, se tiene

$$
\mathbf{U}_R=\begin{pmatrix}1\\a\end{pmatrix},
\qquad
\mathbf{U}_I=\begin{pmatrix}0\\b\end{pmatrix}.
$$

Las soluciones complejas conjugadas producen las soluciones reales

$$
\mathbf{z}_1(t)=\mathbf{Y}_1(t)+\mathbf{Y}_2(t)
=2e^{at}\left(\mathbf{U}_R\cos(bt)-\mathbf{U}_I\sin(bt)\right),
$$

$$
\mathbf{z}_2(t)=-i\left(\mathbf{Y}_1(t)-\mathbf{Y}_2(t)\right)
=2e^{at}\left(\mathbf{U}_R\sin(bt)+\mathbf{U}_I\cos(bt)\right).
$$

Cada vector $$\mathbf{z}_i(t)=(u_i(t),v_i(t))^T$$ se grafica en el plano posición–velocidad, con $$u_i$$ en el eje horizontal y $$v_i$$ en el vertical. Se comparan $$c=1$$ y $$c=\tfrac12$$. El intervalo computacional es finito, pero suficientemente largo para aproximar $$t\to\infty$$.

#### Parámetros y puntos iniciales

Para $$c=1$$,

$$
a=-\frac12,\qquad b=\frac{\sqrt3}{2},\qquad
\mathbf{z}_1(0)=\begin{pmatrix}2\\-1\end{pmatrix},\quad
\mathbf{z}_2(0)=\begin{pmatrix}0\\\sqrt3\end{pmatrix}.
$$

Para $$c=\tfrac12$$,

$$
a=-\frac14,\qquad b=\frac{\sqrt{15}}{4},\qquad
\mathbf{z}_1(0)=\begin{pmatrix}2\\-\tfrac12\end{pmatrix},\quad
\mathbf{z}_2(0)=\begin{pmatrix}0\\\tfrac{\sqrt{15}}{2}\end{pmatrix}.
$$

En los dos casos, $$a<0$$ y por ello $$\mathbf{z}_1(t),\mathbf{z}_2(t)\to(0,0)^T$$ cuando $$t\to\infty$$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Soluciones reales derivadas del par de modos complejos conjugados
def soluciones_reales(c, t):
    a = -c / 2
    b = np.sqrt(4 - c**2) / 2
    U_R = np.array([1.0, a])
    U_I = np.array([0.0, b])

    cos_bt = np.cos(b * t)[:, None]
    sin_bt = np.sin(b * t)[:, None]
    envolvente = 2 * np.exp(a * t)[:, None]

    z1 = envolvente * (U_R * cos_bt - U_I * sin_bt)
    z2 = envolvente * (U_R * sin_bt + U_I * cos_bt)
    return z1, z2, a, b, U_R, U_I

# Se usa un intervalo largo para aproximar el límite t -> infinito
t = np.linspace(0.0, 32.0, 1800)
casos_c = [1.0, 0.5]
colores = {"z1": "#2878B5", "z2": "#E07A32"}

fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

for ax, c_actual in zip(axes, casos_c):
    z1, z2, a, b, U_R, U_I = soluciones_reales(c_actual, t)

    print(f"c = {c_actual:g}: a = {a:.6f}, b = {b:.6f}")
    print(f"  U_R = {U_R}, U_I = {U_I}")
    print(f"  z1(0) = {z1[0]}, z2(0) = {z2[0]}")
    print(f"  z1(32) = {z1[-1]}, z2(32) = {z2[-1]}")

    for nombre, z, color in ((r"$z_1(t)$", z1, colores["z1"]),
                             (r"$z_2(t)$", z2, colores["z2"])):
        ax.plot(z[:, 0], z[:, 1], color=color, linewidth=2.2, label=nombre)
        ax.scatter(z[0, 0], z[0, 1], s=65, color=color, edgecolor="white",
                   linewidth=1.0, zorder=4)

        # Flechas para indicar la dirección en la que aumenta t
        for idx in (100, 420, 740, 1060, 1380, 1600):
            ax.annotate(
                "",
                xy=z[idx + 18, :],
                xytext=z[idx, :],
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.15,
                                mutation_scale=12),
                zorder=3,
            )

    ax.scatter(0, 0, marker="*", s=125, color="#263238", edgecolor="white",
               linewidth=0.8, zorder=5, label="Equilibrio")
    ax.axhline(0, color="black", linewidth=0.75, alpha=0.65)
    ax.axvline(0, color="black", linewidth=0.75, alpha=0.65)
    ax.set_xlim(-2.15, 2.15)
    ax.set_ylim(-2.15, 2.15)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"Posición, $u$")
    ax.set_ylabel(r"Velocidad, $v=\dot{u}$")
    ax.set_title(rf"Trayectorias para $c={c_actual:g}$")
    ax.grid(True, which="major", linewidth=0.7, alpha=0.55)
    ax.minorticks_on()
    ax.grid(True, which="minor", linewidth=0.35, alpha=0.3)
    ax.legend(loc="upper right", frameon=True)

fig.suptitle(r"Soluciones reales $z_1(t)$ y $z_2(t)$ en el plano posición–velocidad", fontsize=14)
plt.show()

#### Interpretación de las gráficas

Las curvas comienzan en los puntos indicados y giran alrededor del equilibrio. La envolvente $$e^{-ct/2}$$ disminuye con el tiempo, de modo que ambas soluciones espiralan hacia el origen. Para $$c=\tfrac12$$ la envolvente decae más lentamente que para $$c=1$$, por lo que la trayectoria tarda más en acercarse al equilibrio.